# 02. Валидация каскада

Этот ноутбук проверяет каскад как бинарный классификатор на detection-датасете в YOLO-формате.

Бинарное правило:

- `ACCEPT` + в GT-разметке есть хотя бы один bbox = `TP`
- `ACCEPT` + GT-разметка пустая = `FP`
- `REJECT` + GT-разметка пустая = `TN`
- `REJECT` + в GT-разметке есть хотя бы один bbox = `FN`

In [ ]:
from pathlib import Path
import csv
import json
import sys
from collections import Counter

import pandas as pd
import yaml
from tqdm.auto import tqdm

VALIDATOR_ROOT = Path.cwd()
if not (VALIDATOR_ROOT / "config.yaml").exists():
    VALIDATOR_ROOT = VALIDATOR_ROOT.parent

sys.path.insert(0, str(VALIDATOR_ROOT))
CONFIG_PATH = VALIDATOR_ROOT / "config.yaml"
OUTPUT_DIR = VALIDATOR_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print(VALIDATOR_ROOT)


Загружаем конфиг. Для быстрой проверки можно поставить маленькое значение `MAX_IMAGES`.

In [ ]:
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
DATASET_ROOT = Path(config["dataset"]["root"])
SPLIT = config["dataset"].get("split", "val")
MAX_IMAGES = None

print("dataset:", DATASET_ROOT)
print("split:", SPLIT)


Создаем каскад из `config.yaml`. Эта ячейка тяжелая, потому что загружает ONNX-веса.

In [ ]:
from src.cascade import CascadeFilter, load_settings

settings = load_settings(CONFIG_PATH)
cascade = CascadeFilter(settings)
print("cascade loaded")


Собираем samples и проверяем баланс GT.

In [ ]:
from src.yolo_dataset import iter_yolo_samples

samples = iter_yolo_samples(DATASET_ROOT, SPLIT, max_images=MAX_IMAGES)
print("images:", len(samples))
print("gt positives:", sum(s.has_gt_bbox for s in samples))
print("gt negatives:", sum(not s.has_gt_bbox for s in samples))


Запускаем валидацию. CSV по каждой картинке нужен, чтобы сортировать ошибки и открывать конкретные файлы.

In [ ]:
from src.metrics import confusion_bucket, compute_binary_metrics

rows = []
counts = Counter()
reason_counts = Counter()

for sample in tqdm(samples):
    result = cascade.run_path(sample.image_path)
    bucket = confusion_bucket(result["decision"], sample.has_gt_bbox)
    counts[bucket] += 1
    reason_counts[result["reason"]] += 1

    rows.append({
        "image_path": str(sample.image_path),
        "label_path": str(sample.label_path),
        "has_gt_bbox": sample.has_gt_bbox,
        "decision": result["decision"],
        "reason": result["reason"],
        "bucket": bucket,
        "garment_count": len(result["garment_detections"]),
        "bad_class_count": len(result["bad_class_detections"]),
        "bad_classes": ",".join(sorted({d.get("class_name", "") for d in result["bad_class_detections"]})),
        "garment_max_conf": max([d["confidence"] for d in result["garment_detections"]], default=0.0),
        "bad_max_conf": max([d["confidence"] for d in result["bad_class_detections"]], default=0.0),
        "total_ms": result.get("timings_ms", {}).get("total", 0.0),
    })

metrics = compute_binary_metrics(counts)
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("reasons:", dict(reason_counts))


Сохраняем и смотрим таблицу результатов.

In [ ]:
df = pd.DataFrame(rows)
out_csv = OUTPUT_DIR / f"cascade_validation_{SPLIT}.csv"
df.to_csv(out_csv, index=False, encoding="utf-8")

print(out_csv)
display(df.head(20))
display(df.groupby(["bucket", "reason"]).size().reset_index(name="count"))


Смотрим самые полезные группы ошибок. Обычно основные эксперименты с thresholds и class rules идут по `FP` и `FN`.

In [ ]:
display(df[df["bucket"].isin(["FP", "FN"])].sort_values(["bucket", "reason", "garment_max_conf"], ascending=[True, True, False]).head(50))


Визуализируем несколько ошибок. Синий = bbox одежды, желтый = плохие классы, из-за которых сработал reject.

In [ ]:
from src.yolo_onnx import read_image_rgb
from src.visualization import draw_cascade_result, show_image

ERROR_BUCKET = "FN"  # можно поставить FP или FN
N_SHOW = 5

for _, row in df[df["bucket"] == ERROR_BUCKET].head(N_SHOW).iterrows():
    image_path = Path(row["image_path"])
    result = cascade.run_path(image_path)
    image_rgb = read_image_rgb(image_path)
    drawn = draw_cascade_result(image_rgb, result)
    show_image(drawn, title=f'{row["bucket"]}: {image_path.name} / {row["reason"]}')
